In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import from_json, col, struct, to_json
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType, StringType

In [ ]:
spark = SparkSession.builder \
    .appName("FraudDetection") \
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0") \
    .getOrCreate()

In [ ]:
users_df = spark.read.csv("data/user_table.csv", header=True, inferSchema=True)

In [ ]:
tx_schema = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("userId", IntegerType(), True),
    StructField("amount", DoubleType(), True),
    StructField("timestamp", DoubleType(), True)
])

kafka_stream = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("subscribe", "fraud-detection") \
    .load()

In [ ]:
parsed_stream = kafka_stream.select(from_json(col("value").cast("string"), tx_schema).alias("tx")).select("tx.*")
fraud_stream = parsed_stream.filter(col("amount") > 10000.0)

In [ ]:
enriched_fraud = fraud_stream.join(users_df, "userId")

In [ ]:
output_stream = enriched_fraud \
    .withColumn("value", to_json(struct("*")).cast("string")) \
    .select("value")

In [ ]:
query = output_stream.writeStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:9092") \
    .option("topic", "fraud-notification") \
    .option("checkpointLocation", "/workspace/checkpoints_2") \
    .start()
query.awaitTermination()